# Extraction des Linéaments et Failles Géologiques

Ce notebook utilise des filtres de détection de contours (Sobel) sur les caractéristiques Prithvi pour extraire les structures géologiques.

In [ ]:
!pip install geemap earthengine-api rasterio terratorch torch opencv-python matplotlib -q
import ee, geemap, torch, rasterio, cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from terratorch import BACKBONE_REGISTRY

ee.Initialize(project='geocongoai-api')

In [ ]:
roi = ee.Geometry.Rectangle([15.4, -4.6, 15.8, -4.2])
image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate('2023-01-01', '2023-12-31').median().clip(roi)
geemap.ee_export_image(image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), 'input.tif', scale=30, region=roi)

In [ ]:
model = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", num_frames=1, in_chans=6, pretrained=True).eval().to('cpu')
with rasterio.open('input.tif') as src: img = src.read().astype(np.float32) / 10000.0
with torch.no_grad():
    out = model(torch.from_numpy(img).unsqueeze(0))
    feats = out[0] if isinstance(out, list) else out
    
h_feat = int(np.sqrt(feats.shape[1]-1))
pca = PCA(n_components=1)
pca_img = pca.fit_transform(feats[0, 1:].numpy()).reshape(h_feat, -1)
pca_norm = ((pca_img - pca_img.min()) / pca_img.ptp() * 255).astype(np.uint8)

edges = cv2.Canny(pca_norm, 50, 150)
plt.imshow(edges, cmap='gray')
plt.title("Linéaments et Failles Géologiques")
plt.show()